# OneComp Hands-On


### Prerequisites

- Python 3.12+ (OneComp requirement)
- CUDA GPU (8GB+ VRAM recommended; we pick a model that fits Colab T4)
- `onecomp >= 1.2.0`

> For Colab, step-by-step instructions are in comments in the first setup cell.

## 0. Setup

Install OneComp if it is not already installed. On Colab, uncomment the lines in the cell below.

In [19]:
# Enable only on first run in Colab.
# Select a Python 3.12 runtime before running.
# !pip install -q "onecomp[cu130,vllm]" matplotlib

%matplotlib inline

import onecomp
print(f"onecomp version: {onecomp.__version__}")

## 1. Select the model

We use **`TinyLlama/TinyLlama-1.1B-Chat-v1.0`** in this tutorial.

Why this model:

- **Small** (1.1B / ~2.2GB in FP16) → runs comfortably on Colab T4
- **No gating** → no Hugging Face login; viewers can try immediately
- **Chat-tuned** → the chat demo at the end works out of the box

> Swap the model ID here to try other Llama / Qwen3 / Gemma models.

In [21]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
fp16_size_gb = n_params * 2 / 1024**3

print(f"model:     {MODEL_ID}")
print(f"device:    {DEVICE}")
print(f"params:    {n_params/1e9:.2f} B")
print(f"FP16 size: {fp16_size_gb:.2f} GB")

In [22]:
# FP16 output (compare side-by-side with quantized output later)
PROMPT = "In one sentence, what is post-training quantization?"

messages = [{"role": "user", "content": PROMPT}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(DEVICE)
input_len = inputs["input_ids"].shape[1]
print(PROMPT)
print(inputs)

with torch.no_grad():
    out_ids = model.generate(**inputs, max_new_tokens=80, do_sample=False)

fp16_response = tokenizer.decode(out_ids[0][input_len:], skip_special_tokens=True)
print("=== FP16 (before quantization) output ===")
print(fp16_response)

## 2. 4bit RTN

In [48]:
target_linear = model.model.layers[0].self_attn.q_proj
W = target_linear.weight.detach().clone().float().cpu()

print(f"layer:    model.layers[0].self_attn.q_proj")
print(f"shape:    {tuple(W.shape)}  (out_features, in_features)")
print(f"dtype:    {W.dtype}")
print(f"|W|.max:  {W.abs().max().item():.4f}")
print(f"|W|.mean: {W.abs().mean().item():.4f}")

In [35]:
import torch
import torch.nn as nn
import transformers

# link) https://github.com/FujitsuResearch/OneCompression/blob/main/onecomp/quantizer/rtn/rtn_impl.py 

def quantize(
    x: torch.Tensor,
    scale: torch.Tensor,
    zero_point: torch.Tensor,
    q_min: int,
    q_max: int,
) -> torch.Tensor:
    """Quantize floating-point values to integers.

    Computes ``clamp(round(x / scale) + zero_point, q_min, q_max)``.

    Args:
        x: Input tensor (floating-point).
        scale: Scale coefficient.
        zero_point: Zero point.
        q_min: Minimum quantization level.
        q_max: Maximum quantization level.

    Returns:
        Quantized integer tensor (clamped to the range [q_min, q_max]).
    """
    w_int = torch.clamp(torch.round(x / scale) + zero_point, q_min, q_max).int()
    return w_int


def dequantize(
    quantized: torch.Tensor,
    scale: torch.Tensor,
    zero_point: torch.Tensor,
) -> torch.Tensor:
    """Dequantize integer values back to floating-point.

    Args:
        quantized: Quantized integer tensor.
        scale: Scale coefficient.
        zero_point: Zero point.

    Returns:
        Dequantized floating-point tensor.
    """
    return (quantized.float() - zero_point) * scale


def pseudo_quantize_tensor(
    w: torch.Tensor,
    n_bit: int = 8,
    q_group_size: int = -1,
    zero_point: bool = True,
    inplace: bool = False,
    perchannel: bool = True,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Pseudo-quantize a tensor using the Round-To-Nearest method.

    Args:
        w: Weight tensor to quantize.
        n_bit: Number of quantization bits.
        q_group_size: Group size (-1 means the entire row).
        zero_point: If True, asymmetric quantisation. If False,
            symmetric quantisation (min/max symmetrised around zero).
        inplace: Whether to perform in-place operations.
        perchannel: If True, compute one scale/zero per output channel
            (row). If False, use a single scale/zero for the entire
            tensor (per-tensor). Ignored when ``q_group_size > 0``.

    Returns:
        w_quant: Dequantized weights (floating-point).
        scale: Scale coefficient.
        zero_point_val: Zero point.
        w_int: Quantized weights (integer values).
    """
    if not inplace:
        w = w.clone()

    # Save the original shape
    org_w_shape = w.shape

    # Configure group size
    if q_group_size > 0:
        # (out_features, in_features) -> (out_features, num_groups, group_size)
        if w.shape[-1] % q_group_size != 0:
            raise ValueError(
                f"Tensor shape {w.shape[-1]} must be divisible by group size {q_group_size}"
            )
        w = w.reshape(-1, w.shape[-1] // q_group_size, q_group_size)
    elif perchannel:
        # Treat the entire row as a single group
        w = w.reshape(-1, 1, w.shape[-1])
    else:
        # Per-tensor: single scale/zero for the entire weight
        w = w.flatten().reshape(1, 1, -1)

    # Quantization levels: always unsigned [0, 2^n_bit - 1]
    sym = not zero_point
    q_max = 2**n_bit - 1
    q_min = 0

    # Compute min and max values per group (with zero included in the range)
    tmp = torch.zeros(1, device=w.device)
    w_max = torch.maximum(w.amax(dim=-1, keepdim=True), tmp)
    w_min = torch.minimum(w.amin(dim=-1, keepdim=True), tmp)

    # Symmetric: symmetrise range around zero
    if sym:
        w_max = torch.maximum(torch.abs(w_min), w_max)
        w_min = -w_max

    # Handle all-zero groups
    dead = (w_min == 0) & (w_max == 0)
    w_min[dead] = -1
    w_max[dead] = +1

    # Compute scale and zero point
    scale = ((w_max - w_min) / q_max).clamp(min=1e-5)

    if sym:
        zero_point_val = torch.full_like(scale, (q_max + 1) / 2)
    else:
        zero_point_val = torch.round(-w_min / scale)

    # Quantize and dequantize
    w_int = quantize(w, scale, zero_point_val, q_min, q_max)
    w_quant = dequantize(w_int, scale, zero_point_val)

    # Restore original shape
    w_quant = w_quant.reshape(org_w_shape)
    w_int = w_int.reshape(org_w_shape)
    scale = scale.squeeze(-1)
    zero_point_val = zero_point_val.squeeze(-1)

    return w_quant, scale, zero_point_val, w_int


In [47]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

W_np = W.flatten().numpy()

W_dq, _, _, _ = pseudo_quantize_tensor(W, n_bit=4, perchannel=False, zero_point=False)
W_dq_np = W_dq.flatten().numpy()
err_np = (W - W_dq).flatten().numpy()

vmax = float(np.max(np.abs(W_np)))
bins = np.linspace(-vmax, vmax, 201)

fig, axes = plt.subplots(2, 1, figsize=(8, 8))

# (1) Overlay FP16 vs dequantized on log-y so the tails are visible
axes[0].hist(W_np, bins=bins, color="steelblue", alpha=0.55, label="FP16 (original)")
axes[0].hist(W_dq_np, bins=bins, color="darkorange", alpha=0.55, label="4bit RTN dequantized")
axes[0].set_yscale("log")
axes[0].set_title("Weight distribution (log y)")
axes[0].set_xlabel("weight value")
axes[0].set_ylabel("count (log)")
axes[0].legend(loc="upper right")

# (2) Error distribution, log-y to see the tails clearly
err_max = float(np.max(np.abs(err_np)))
axes[1].hist(err_np, bins=np.linspace(-err_max, err_max, 201), color="seagreen", alpha=0.8)
axes[1].set_yscale("log")
axes[1].set_title("Quantization error  W − Ŵ  (log y)")
axes[1].set_xlabel("error")
axes[1].set_ylabel("count (log)")

fig.tight_layout()
# display(fig)

In [ ]:
print(W_dq[0][:10])
print(W[0][:10])


## 3. One-line quantization with OneComp

`Runner.auto_run()` runs the full pipeline automatically:

1. Load model and tokenizer from Hugging Face
2. Detect GPU VRAM and run AutoBit
3. Quantize with GPTQ + QEP
4. Evaluate perplexity
5. Save the quantized model to disk

From the CLI: one line — `onecomp TinyLlama/TinyLlama-1.1B-Chat-v1.0`. In Python: the line below.

In [32]:
# Free the FP16 model from VRAM before quantization
import gc

del model
gc.collect()
torch.cuda.empty_cache()

In [18]:
from onecomp import Runner, setup_logger

setup_logger()

SAVE_DIR = "./tinyllama-chat-onecomp"

runner = Runner.auto_run(
    model_id=MODEL_ID,
    save_dir=SAVE_DIR,
    total_vram_gb=1,
    evaluate=False,
)

In [13]:
import os
SAVE_DIR = "./tinyllama-chat-onecomp"

def dir_size_gb(path: str) -> float:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024**3


quantized_size_gb = dir_size_gb(SAVE_DIR)
print(f"FP16 model size:     {fp16_size_gb:.2f} GB")
print(f"Quantized model size: {quantized_size_gb:.2f} GB")
print(f"Compression ratio: {fp16_size_gb / quantized_size_gb:.2f}x")

In [42]:
original_ppl, _, quantized_ppl = runner.calculate_perplexity(
    original_model=True,
    quantized_model=True,
)
ppl_delta_pct = (quantized_ppl - original_ppl) / original_ppl * 100

print(f"FP16 perplexity      (wikitext-2): {original_ppl:.3f}")
print(f"Quantized perplexity (wikitext-2): {quantized_ppl:.3f}")
print(f"Δ:                                  {ppl_delta_pct:+.2f}%  "
      f"(lower PPL is better)")

In [15]:
# Free the Runner from VRAM too (make room for the next vLLM load)
del runner
gc.collect()
torch.cuda.empty_cache()

## 4. Run the quantized model with vLLM

OneComp provides a **vLLM plugin** so you can load saved quantized models directly in vLLM.

### Notes for Colab

- Running a full **`vllm serve` (HTTP server)** on Colab is brittle (ports, process management)
- Here we use **`vllm.LLM` offline inference** (load in-process and call `.generate()` / `.chat()`)
- **A ChatGPT-style UI with `vllm serve` + Open WebUI**

> The plugin registers automatically via `entry_points` when you `pip install onecomp[vllm]`; no extra vLLM config is needed.

In [16]:
from vllm import LLM, SamplingParams
SAVE_DIR = "./tinyllama-chat-onecomp"
llm = LLM(
    model=SAVE_DIR,
    max_model_len=1024,
    dtype="float16",
    enforce_eager=True,
    gpu_memory_utilization=0.80,
)

sampling = SamplingParams(max_tokens=80, temperature=0.0)

outputs = llm.chat(
    [{"role": "user", "content": PROMPT}],
    sampling,
    use_tqdm=False,
    chat_template_content_format="string",  
)
quantized_response = outputs[0].outputs[0].text
print("=== FP16 (before quantization) ===")
print(fp16_response.strip())
print()
print("=== 4-bit OneComp (after quantization) ===")
print(quantized_response.strip())

## 5. Try a chat

vLLM's `LLM.chat()` accepts OpenAI-style `messages`. Keep multi-turn history and chat with the quantized Llama.

In [49]:
chat_sampling = SamplingParams(max_tokens=200, temperature=0.2)

conversation = [
    {"role": "system", "content": "You are a helpful assistant who answers concisely."},
]


def ask(user_message: str) -> str:
    conversation.append({"role": "user", "content": user_message})
    out = llm.chat(conversation, chat_sampling, use_tqdm=False)
    reply = out[0].outputs[0].text.strip()
    conversation.append({"role": "assistant", "content": reply})
    return reply


print("USER:", "What is Fujitsu OneComp in one sentence?")
print("BOT :", ask("What is Fujitsu OneComp in one sentence?"))
print()
print("USER:", "And which quantization methods does it support?")
print("BOT :", ask("And which quantization methods does it support?"))